In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Change only these if your Unity Catalog names are different
CATALOG = f"formula1_{env}"
BRONZE = "bronze"
SILVER = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

def bronze_table(name):
    return f"{CATALOG}.{BRONZE}.{name}"

def silver_table(name):
    return f"{CATALOG}.{SILVER}.{name}"

In [0]:
print(CATALOG)

In [0]:
races = spark.table(bronze_table("races"))
races.printSchema()
print("Bronze rows:",races.count())

## 1. NULL + duplicate checks

In [0]:
display(races.filter(
    F.col("race_id").isNull() |
    F.col("circuit_id").isNull()
))
display(races.groupBy("race_id").count().filter(F.col("count")>1))

## 2. String cleansing

In [0]:
races_clean = (
    races
    .filter(F.col("race_id").isNotNull())
    .dropDuplicates(["race_id"])
    .withColumn("race_name_clean",F.trim("name"))
    .withColumn("race_name_lower",F.lower("race_name_clean"))
    .withColumn("name_length",F.length("race_name_clean"))
    .withColumn("name_prefix",F.substring("race_name_clean",1,5))
    .withColumn("name_parts",F.split("race_name_clean"," ")) # SPLIT ON THE SPACE
    .withColumn("name_parts_count",F.size("name_parts"))
    .withColumn("name_parts_first",F.element_at("name_parts",1))
    .withColumn("name_parts_last",F.element_at("name_parts",F.size("name_parts"))) # LAST ELEMENT IN THE ARRAY 
)
display(races_clean.select(
    "race_id","name","race_name_clean","race_name_lower",
    "name_length","name_prefix","name_parts","name_parts_count","name_parts_first","name_parts_last"
).limit(20))

## 3. contains / startsWith / endsWith

In [0]:
display(races_clean.filter(F.col("race_name_lower").contains("grand"))
                  .select("race_id","race_name_clean").limit(20))
display(races_clean.filter(F.col("race_name_lower").startswith("a"))
                  .select("race_id","race_name_clean").limit(20))
display(races_clean.filter(F.col("race_name_lower").endswith("a"))
                  .select("race_id","race_name_clean").limit(20))

## 4. Date + timestamp functions

In [0]:
races_clean = (
    races_clean
    .withColumn("year_from_date",F.year("date"))
    .withColumn("month",F.month("date"))
    .withColumn("day",F.dayofmonth("date"))
    .withColumn("year_month",F.date_format("date","yyyy-MM"))
    .withColumn("days_from_today",F.datediff(F.current_date(),"date"))
    .withColumn("date_plus_7",F.date_add("date",7))
    .withColumn("race_timestamp_date",F.to_date("race_timestamp"))
    .withColumn("race_hour",F.hour("race_timestamp"))
)
display(races_clean.select(
    "race_id","race_year","date","year_from_date","month","day",
    "year_month","days_from_today","date_plus_7",
    "race_timestamp","race_hour"
).limit(20))

## 5. CASE statement

In [0]:
races_clean = races_clean.withColumn(
    "season_period",
    F.when(F.col("race_year") < 2000,"Historical")
     .when(F.col("race_year") < 2010,"2000-2009")
     .when(F.col("race_year") < 2020,"2010-2019")
     .otherwise("2020+")
)
display(races_clean.select("race_year","season_period"))
# display(races_clean.select("race_year","season_period").orderBy("race_year").limit(20))

## 6. Filter + GroupBy + OrderBy

In [0]:
display(
    races_clean.filter(F.col("race_year").isNotNull())
               .orderBy(F.desc("date")).limit(20)
)
display(
    races_clean.groupBy("race_year").count().orderBy("race_year")
)

## 7. Write Silver

In [0]:
races_silver = races_clean.select(
    F.col("race_id").cast("int"),
    F.col("race_year").cast("int"),
    F.col("round").cast("int"),
    F.col("circuit_id").cast("int"),
    "race_name_clean",
    "ingestion_date",
    "race_timestamp",
    "date",
    "season_period"
)

races_silver.printSchema()

(
    races_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(silver_table("races"))
)
print("Created:",silver_table("races"))

In [0]:
silver=spark.table(silver_table("races"))
print("Silver rows:",silver.count())
display(silver.limit(20))